# IntZ Example 13: Kinematic State vs Circular Velocity

**EPS Research IntZ Kinematic Corpus v1.0** | Flynn, D.C. (2026)

V/σ ratio vs circular velocity for KROSS Tier-1 galaxies — probing
whether faster rotators are more kinematically settled.

> **Note (August 2026):** This notebook previously showed omega vs Vc.
> Omega values have been withdrawn — see
> [CORRECTIONS.md](../../CORRECTIONS.md). Replaced with V/σ vs Vc.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv, numpy as np

# Load IntZ corpus flat CSV
rows = []
with open('intz_corpus_v1b_flat.csv') as f:
    for row in csv.DictReader(f):
        def fv(k):
            try: return float(row[k])
            except: return None
        rows.append({
            'z':      fv('z_spec'),
            'Vc':     fv('Vc_kms'),
            'sigma':  fv('sigma0_kms'),
            'vos':    fv('v_over_sigma'),
            'log_ms': fv('log_mstar'),
            'Reff':   fv('Reff_kpc'),
            'survey': row['survey'],
            'tier':   row['quality_tier'],
        })

kross1 = [r for r in rows if r['survey']=='KROSS' and r['tier']=='1' and r['Vc']]
print(f'KROSS Tier-1 with Vc: {len(kross1)} galaxies')


In [ ]:
kross_vos = [r for r in kross1 if r['vos'] and 0 < r['vos'] < 25]
vc  = [r['Vc']  for r in kross_vos]
vos = [r['vos'] for r in kross_vos]
z   = [r['z']   for r in kross_vos]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ax = axes[0]
sc = ax.scatter(vc, vos, c=z, cmap='plasma', s=15, alpha=0.6, vmin=0.6, vmax=1.05)
plt.colorbar(sc, ax=ax, label='Redshift z')
ax.axhline(1.0, color='black', lw=1.5, ls='--', alpha=0.5, label='V/σ = 1')
ax.set_xlabel('Vc (km/s)', fontsize=12)
ax.set_ylabel('V/σ', fontsize=12)
ax.set_title('Kinematic State vs Circular Velocity\nKROSS Tier-1', fontsize=11)
ax.legend(fontsize=9)

# Binned median
vc_arr  = np.array(vc)
vos_arr = np.array(vos)
vc_bins = np.percentile(vc_arr, [0, 25, 50, 75, 100])
vc_mid, vos_med, vos_err = [], [], []
for i in range(len(vc_bins)-1):
    mask = (vc_arr >= vc_bins[i]) & (vc_arr < vc_bins[i+1])
    if mask.sum() > 3:
        vc_mid.append(np.median(vc_arr[mask]))
        vos_med.append(np.median(vos_arr[mask]))
        vos_err.append(np.std(vos_arr[mask])/np.sqrt(mask.sum()))

ax2 = axes[1]
ax2.bar(range(len(vc_mid)), vos_med,
        color='#ff7f0e', alpha=0.8, edgecolor='white')
ax2.errorbar(range(len(vc_mid)), vos_med, yerr=vos_err,
             fmt='none', color='black', capsize=5)
ax2.axhline(1.0, color='black', lw=1.5, ls='--', alpha=0.5)
ax2.set_xticks(range(len(vc_mid)))
ax2.set_xticklabels([f'Q{i+1}
({vc_bins[i]:.0f}–{vc_bins[i+1]:.0f})' 
                     for i in range(len(vc_mid))], fontsize=9)
ax2.set_ylabel('Median V/σ', fontsize=12)
ax2.set_title('V/σ by Vc Quartile\nKROSS Tier-1', fontsize=11)

plt.suptitle('IntZ Corpus v1.0 — Kinematic State vs Circular Velocity\n'
             f'KROSS Tier-1 (N={len(kross_vos)})', fontsize=11)
plt.tight_layout()
plt.savefig('intz_nb13_vos_vs_vc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'N galaxies with Vc + V/σ: {len(kross_vos)}')
print(f'Median V/σ: {np.median(vos):.2f}')
print(f'Fraction with V/σ > 1: {sum(v>1 for v in vos)/len(vos)*100:.0f}%')
